In [0]:
%sql
-- Customer retention cohort analysis
-- Business question: of customers who first purchased in month X,
--   what % are still active N months later?
-- Identity = customer_unique_id (person), NOT customer_id (per-order in Olist).
-- Delivered orders only. Grain: one row per (cohort_month, month_offset).
-- retention_rate = active_customers / cohort's offset-0 size * 100.
WITH monthly_activity AS (
    SELECT DISTINCT
        dc.customer_unique_id,
        dd.year_month AS activity_month,
        dd.year  AS activity_year,
        dd.month AS activity_month_num
    FROM gold.fact_sales fs
    JOIN gold.dim_customer dc ON fs.customer_sk = dc.customer_sk
    JOIN gold.dim_date     dd ON fs.date_key   = dd.date_key
    WHERE fs.order_status = 'delivered'
),
cohorted AS (
    SELECT
        customer_unique_id,
        activity_month,
        activity_year,
        activity_month_num,
        -- cohort = first month this person was ever active
        MIN(activity_month) OVER (PARTITION BY customer_unique_id) AS cohort_month,
        -- absolute month index for THIS activity
        (activity_year * 12 + activity_month_num) AS activity_index
    FROM monthly_activity
),
offsets AS (
    SELECT 
        customer_unique_id,
        activity_index  - MIN(activity_index) OVER (PARTITION BY customer_unique_id) AS offset_cohort,
        cohort_month,
        activity_index
    FROM cohorted
    ORDER BY customer_unique_id, activity_month
)
,
retentation AS (
    SELECT 
        cohort_month,
        offset_cohort,
        COUNT(DISTINCT customer_unique_id) AS active_customers
    FROM offsets
    GROUP BY cohort_month, offset_cohort
    ORDER BY cohort_month, offset_cohort
)
SELECT 
    *, 
    ROUND(100 * (active_customers / MAX(active_customers) OVER (PARTITION BY cohort_month)), 5) AS retentation_rate
FROM retentation
ORDER BY cohort_month, offset_cohort



